In [1]:
import numpy as np
import pandas as pd

In [ ]:
import urllib.request

url = "https://www.gutenberg.org/files/1268/1268-0.txt"

with urllib.request.urlopen(url) as response:
    text = response.read().decode("utf-8")

with open("mysterious_island.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [26]:
with open('mysterious_island.txt','r') as fp:
    text = fp.read()

text = text.replace("\r", "")

In [27]:
start_indx = text.find('THE MYSTERIOUS ISLAND')
end_idx = text.find('End of the Project Gutenberg')
text = text[start_indx:end_idx]
char_set = set(text)
print(f'Total Length: {len(text)}')
print(f'Unique Characters: {len(char_set)}')

Total Length: 1112310
Unique Characters: 80


## **Building the dictionary to map characters to integers**

In [28]:
np.array(char_set)

array({'Q', 't', '’', '=', 'b', '1', 'R', 'F', 's', 'T', 'y', '*', ';', '.', 'k', '8', 'J', 'j', 'B', 'h', ')', 'D', 'Y', '!', 'P', 'Z', '?', '‘', 'H', 'a', 'C', 'W', '(', 'x', '5', '/', '0', '&', 'c', 'm', '9', '3', 'K', '”', 'v', 'i', 'w', 'n', 'N', 'o', 'g', 'u', 'l', 'V', '“', 'L', '4', 'z', ',', 'd', 'O', ' ', '-', 'e', '2', 'M', ':', 'r', 'I', 'U', 'S', 'E', 'q', 'A', '7', 'p', 'f', '\n', '6', 'G'},
      dtype=object)

<span style='color: chocolate'> 1) sort </span>

In [29]:
chars_sorted = sorted(char_set)

<span style='color: chocolate'> 2. dictionary for character and key </span>

In [30]:
char2int = {ch:i for i, ch in enumerate(chars_sorted)}

<span style='color: chocolate'> 3. .array() for reverse </span>

In [31]:
char_array = np.array(chars_sorted)

<span style='color: chocolate'> 4. encode all characters loop </span>

In [32]:
text_encoded = np.array([char2int[ch] for ch in text], dtype=np.int32)

In [33]:
print('Text encoded shape: ', text_encoded.shape)

Text encoded shape:  (1112310,)


In [34]:
print(text[:15], '==Encoding ==>', text_encoded[:15])

THE MYSTERIOUS  ==Encoding ==> [44 32 29  1 37 48 43 44 29 42 33 39 45 43  1]


In [36]:
print(text_encoded[15:21], '== Reverse ==>',
      ''.join(char_array[text_encoded[15:21]]))

[33 43 36 25 38 28] == Reverse ==> ISLAND


**Create a TensorFlow dataset**

<span style='color: chocolate'> 1) tf.data.Dataset.from_tensor_slices()</span>

In [37]:
import tensorflow as tf
ds_text_encoded = tf.data.Dataset.from_tensor_slices(text_encoded)

In [39]:
for ex in ds_text_encoded.take(5):
    print('{} -> {}'.format(ex.numpy(), char_array[ex.numpy()]))

44 -> T
32 -> H
29 -> E
1 ->  
37 -> M


2026-07-10 11:30:34.596503: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


<span style='color: chocolate'> 2. ds_test_encoded.batch(chunk_size, drop_remainder=True) </span>

In [ ]:
# split data to batch 
seq_length = 40
chunk_size = seq_length + 1
ds_chunks = ds_text_encoded.batch(chunk_size, drop_remainder=True) # drop the last if size is smaller than 40

<span style='color: chocolate'> 3. ds_chunk.map() </span>

In [41]:
# define the function for spliting x and y
def split_input_target(chunk):
    input_seq = chunk[:-1]
    target_seq = chunk[1:]
    return input_seq, target_seq
ds_sequences = ds_chunks.map(split_input_target)

In [ ]:
for example in ds_sequences.take(2):
    print('Input(x): ', repr(''.join(char_array[example[0].numpy()]))) # repr() is a built-in Python function that returns the 'official' string representation of an object
    print('Target(y): ', repr(''.join(char_array[example[1].numpy()])))

Input(x):  'THE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n1'
Target(y):  'HE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n18'
Input(x):  '74\n\n\n\n\nPART 1--DROPPED FROM THE CLOUDS\n\n'
Target(y):  '4\n\n\n\n\nPART 1--DROPPED FROM THE CLOUDS\n\n\n'


2026-07-10 11:46:48.486644: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


<span style='color: chocolate'> 4) .shuffle & .batch </span>

In [46]:
# divide the dataset into mini-batches
BATCH_SIZE = 64
BUFFER_SIZE = 10000
ds = ds_sequences.shuffle(BUFFER_SIZE, reshuffle_each_iteration=False).batch(BATCH_SIZE)

## **Building a character-level RNN model**

In [49]:
def build_model(vocab_size, embedding_dim, rnn_units):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(None,), dtype=tf.int32),
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.LSTM(rnn_units, return_sequences=True), # rnn_units is the size of the LSTM's hidden state, how many neurons/cells in are in the LSTM layer.
        tf.keras.layers.Dense(vocab_size)
    ])
    return model

charset_size = len(char_array) # total vocabularies
embedding_dim = 256 # embedding dimension per character
rnn_units = 512 # layers of hidden state
tf.random.set_seed(1)
model = build_model(
    vocab_size=charset_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, None, 256)      │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, None, 512)      │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, None, 80)       │        41,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,636,432 (6.24 MB)

 Trainable params: 1,636,432 (6.24 MB)

 Non-trainable params: 0 (0.00 B)

In [54]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)
model.fit(ds, epochs=5)

Epoch 1/5
424/424 ━━━━━━━━━━━━━━━━━━━━ 43s 99ms/step - loss: 1.8883
Epoch 2/5
424/424 ━━━━━━━━━━━━━━━━━━━━ 44s 102ms/step - loss: 1.5302
Epoch 3/5
424/424 ━━━━━━━━━━━━━━━━━━━━ 44s 103ms/step - loss: 1.3876
Epoch 4/5
424/424 ━━━━━━━━━━━━━━━━━━━━ 44s 102ms/step - loss: 1.3107
Epoch 5/5
424/424 ━━━━━━━━━━━━━━━━━━━━ 43s 102ms/step - loss: 1.2604


**Evaluation phase - generatig new text passages**

<span style='color: chocolate'> 1. tf.random.categorical(logits=, num_samples=) </span>

In [60]:
tf.random.set_seed(1)
logits = [[1.0, 1.0, 1.0]]
print('Probabilities: ', tf.math.softmax(logits).numpy()[0])

# tf.random.categorical(): samples class indices from a categorical distribution defined by logits
# In a char-level RNN, it's how you turn the model's output logits into an actual next character
# logits: shape (batch_size, num_classes) — unnormalized log-probabilities
# num_samples: how many samples to draw per row
# Returns: shape (batch_size, num_samples) — sampled class indices
# What it does under the hood:
    # Applies softmax to logits → probability distribution over classes
    # Samples from that distribution (each class picked with its probability)

samples = tf.random.categorical(
    logits=logits, num_samples=10
)
tf.print(samples.numpy())

Probabilities:  [0.33333334 0.33333334 0.33333334]
array([[0, 0, 1, 2, 0, 0, 0, 0, 1, 0]])


In [61]:
tf.random.set_seed(1)
logits = [[1.0, 1.0, 3.0]]
print('Probabilities: ', tf.math.softmax(logits).numpy()[0])

# tf.random.categorical(): samples class indices from a categorical distribution defined by logits
# In a char-level RNN, it's how you turn the model's output logits into an actual next character
# logits: shape (batch_size, num_classes) — unnormalized log-probabilities
# num_samples: how many samples to draw per row
# Returns: shape (batch_size, num_samples) — sampled class indices
# What it does under the hood:
    # Applies softmax to logits → probability distribution over classes
    # Samples from that distribution (each class picked with its probability)

samples = tf.random.categorical(
    logits=logits, num_samples=10
)
tf.print(samples.numpy())

Probabilities:  [0.10650698 0.10650698 0.78698605]
array([[2, 0, 2, 2, 2, 0, 1, 2, 2, 0]])


<span style='color: chocolate'> 2. sample function to generate next character </span>

In [77]:
def sample(model, starting_str,
           len_generated_text = 500,
           max_input_length=40,
           scale_factor=2.0):
    encoded_input = [char2int[s] for s in starting_str]
    encoded_input = tf.reshape(encoded_input, (1,-1)) # reshape to 1 row, but can be any number of columns. Fit into the model

    # initialize string
    generated_str = starting_str

    # model.reset_states() # clears the internal hidden states of stateful RNN layers
    for i in range(len_generated_text): # total characters can generate
        # predict from the trained model
        logits = model(encoded_input) # return a list of logits, size as 80
        logits = tf.squeeze(logits, 0) # if first dimension with size 1, delete the dimension

        # pick the result based on the logit probability
        scaled_logits = logits * scale_factor # below provides more detail why use scale_factor
        new_char_indx = tf.random.categorical(scaled_logits, num_samples=1) # only randomly pick one value from the list (80 vocabularies) - based on the probability

        # add the new generated character to the list - this is the return result
        new_char_indx = tf.squeeze(new_char_indx)[-1].numpy()
        generated_str += str(char_array[new_char_indx])

        # create new input - according to max_input_length
        new_char_indx = tf.expand_dims([new_char_indx], 0) # add one layer of dimension
        encoded_input = tf.concat(
            [encoded_input, new_char_indx],
            axis=1)
        encoded_input = encoded_input[:, -max_input_length:]

    return generated_str

#### **Why use scale_factor?**

`scale_factor` here is the **inverse temperature** for sampling — it controls how "confident" vs "creative" the generated text is.

Look at what it does: `scaled_logits = logits * scale_factor`, and those scaled logits go into `tf.random.categorical`, which internally applies softmax to turn them into a probability distribution. Multiplying logits before softmax changes the *sharpness* of that distribution.

The math (softmax with scaling):

$$P(i) = \frac{e^{\alpha \cdot z_i}}{\sum_j e^{\alpha \cdot z_j}}$$

where α is your `scale_factor` and z are the raw logits. Three regimes:

- **`scale_factor = 1.0`** (default): sample from the model's native distribution. Balanced.
- **`scale_factor > 1.0`** (e.g. 2.0): amplifies differences between logits → softmax gets peakier → high-probability chars dominate → text is more **conservative, repetitive, "safe"**. In the limit (α → ∞), it becomes greedy argmax.
- **`scale_factor < 1.0`** (e.g. 0.5): flattens differences → distribution gets more uniform → rarer chars get picked more often → text is more **diverse, creative, chaotic**. In the limit (α → 0), it's uniform random.

This is exactly the **temperature** knob you see in every LLM API (OpenAI, Anthropic, etc.), just parameterized as its reciprocal. The standard formulation is:

$$P(i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

so `scale_factor = 1/T`. Raschka calls it `scale_factor` because multiplying is more intuitive to explain than dividing, but functionally it's temperature sampling.

<span style='color: chocolate'> 3. Generate some new text </span>

In [78]:
tf.random.set_seed(1)
print(sample(model, starting_str='The islan'))

The island was under the stranger preserved on the sea of the brig.

“What is not more than the morning the settlers were to see the settlers were object to suppose the lake of the most sight of the morning, the settlers had been since he had not continued to say the engineer, the convicts had been sure of the south and consequence them to be seen trans of the corral. It was not a ship the shore of the submersion of the island supposed to enter the longer and between the interior of the very later, which


<span style='color:chocolate'> 4. test the scale_factor </span>

In [70]:
logits = np.array([[1.0, 1.0, 3.0]])

In [72]:
tf.math.softmax(logits).numpy()

array([[0.10650698, 0.10650698, 0.78698604]])

In [73]:

tf.math.softmax(0.1 * logits).numpy()

array([[0.31042377, 0.31042377, 0.37915245]])

In [74]:
import math
math.exp(0.3) / (math.exp(0.1) + math.exp(0.1) + math.exp(0.3))

0.37915245309398876

In [76]:
math.exp(3) / (math.exp(1) + math.exp(1) + math.exp(3))

0.7869860421615985

## **Understanding language with the Transformer model**